# C01: データ準備（最新HTMLの差分ダウンロード）

**最終更新日**: 2026-04-29 (rev 2)

| 改訂日 | 内容 |
|--------|------|
| 2026-04-29 (rev 2) | `download_race_html()` の失敗理由を詳細化（HTTPステータス/サイズ/no-data文言/馬リンク数で区別）。429/503 はリトライ。最初の数件の失敗詳細をprint。最終統計に内訳を追加 |
| 2026-04-29 (rev 1) | ダウンロードskip判定を「Parquet含有」→「ディスク上HTMLの有無」に変更（race_id ≠ 日付の前提に対応） |

---

このノートブックでは以下の処理を行います：
1. 最新レースHTMLの一括ダウンロード
2. データ確認

**使い方**: 上から順番にセルを実行してください（`Shift + Enter`）


In [ ]:
# == Colab用: GitHubからリポジトリをクローンして準備 ==
!git clone https://github.com/iinumac/keiba_prediction.git
%cd keiba_prediction

# クローンしたディレクトリがPROJECT_ROOTになる
import os
os.environ['PROJECT_ROOT'] = '/content/keiba_prediction'
print("✅ リポジトリのクローン完了。現在のディレクトリ:", os.getcwd())

Cloning into 'keiba_prediction'...
remote: Enumerating objects: 55546, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 55546 (delta 2), reused 9 (delta 1), pack-reused 55533 (from 2)
Receiving objects: 100% (55546/55546), 197.68 MiB | 26.41 MiB/s, done.
Resolving deltas: 100% (55213/55213), done.
Updating files: 100% (55357/55357), done.
/content/keiba_prediction
✅ リポジトリのクローン完了。現在のディレクトリ: /content/keiba_prediction


---
## ステップ1: 環境設定

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from datetime import datetime
import time
import requests
from pathlib import Path

# プロジェクトルート（notebooks/sagemaker/ から2階層上）
PROJECT_ROOT = Path('.').resolve()  # Colab環境ではカレントディレクトリがルート
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# データパス
HTML_DIR = PROJECT_ROOT / 'data' / 'raceHTML'
HTML_DIR.mkdir(parents=True, exist_ok=True)

print("✅ 環境設定完了")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"HTML_DIR: {HTML_DIR}")

✅ 環境設定完了
PROJECT_ROOT: /content/keiba_prediction
HTML_DIR: /content/keiba_prediction/data/raceHTML


---
## ステップ2: 最新レースHTMLの一括ダウンロード

netkeibaから最新のレースHTMLを自動的にダウンロードします。
- 既に存在するHTMLはスキップ
- レースが存在しない場合は次へスキップ
- 当年のみ

In [ ]:
def generate_race_ids(start_year=2010, end_year=None, debug=False):
    """
    レースIDを生成するジェネレータ

    race_id形式: YYYYCCRRDDNN (12桁)
        YYYY: 年 (2010-現在)
        CC: 競馬場コード (01-10)
        RR: 開催回 (01-10)
        DD: 開催日 (01-20)
        NN: レース番号 (01-12)
    """
    from datetime import datetime
    if end_year is None:
        end_year = datetime.now().year

    for year in range(start_year, end_year + 1):
        for venue_code in range(1, 11):  # 01-10
            kaisai = 1
            while kaisai <= 10:  # 01-10（最大10回）
                day = 1
                kaisai_has_races = False
                skip_kaisai_flag = False

                while day <= 20:  # 01-20（最大20日）
                    race_num = 1

                    while race_num <= 12:  # 01-12
                        race_id = f"{year:04d}{venue_code:02d}{kaisai:02d}{day:02d}{race_num:02d}"
                        skip_signal = yield race_id

                        if skip_signal == 'race_found':
                            kaisai_has_races = True
                        elif skip_signal == 'skip_day':
                            break
                        elif skip_signal == 'skip_kaisai':
                            skip_kaisai_flag = True
                            break

                        race_num += 1

                    if skip_kaisai_flag:
                        break
                    day += 1

                if skip_kaisai_flag:
                    kaisai += 1
                    continue

                if not kaisai_has_races:
                    break

                kaisai += 1


# 失敗理由を集計するためのカウンタ（実行ごとにリセット）
failure_reason_counts = {}
failure_first_samples = {}  # 各理由の最初の1件のrace_id, 詳細

def _record_failure(reason, race_id, detail=''):
    failure_reason_counts[reason] = failure_reason_counts.get(reason, 0) + 1
    if reason not in failure_first_samples:
        failure_first_samples[reason] = (race_id, detail)


def download_race_html(race_id, force=False):
    """
    レースHTMLをダウンロード

    戻り値:
      'exists'        : 既にディスクに存在 (skip)
      'success'       : 新規ダウンロード成功
      'not_found'     : netkeibaに該当レースが存在しない
      'rate_limited'  : 429/503 等で取得できず（リトライ後も失敗）
      'error'         : ネットワーク例外/タイムアウト等
    """
    year = race_id[:4]
    output_path = HTML_DIR / year / f"{race_id}.html"

    if output_path.exists() and not force:
        return 'exists'

    output_path.parent.mkdir(parents=True, exist_ok=True)

    url = f"https://db.netkeiba.com/race/{race_id}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }

    def _get_with_retry():
        # 429/503 は短い backoff 後に1回だけリトライ
        for attempt in range(2):
            try:
                resp = requests.get(url, headers=headers, timeout=15)
            except requests.exceptions.Timeout:
                return None, 'timeout'
            except requests.exceptions.RequestException as e:
                return None, f'request_exception: {type(e).__name__}'
            if resp.status_code in (429, 503) and attempt == 0:
                time.sleep(3.0)
                continue
            return resp, None
        return resp, None

    response, exc_reason = _get_with_retry()
    if response is None:
        _record_failure('error', race_id, exc_reason or 'unknown')
        return 'error'

    if response.status_code in (429, 503):
        _record_failure('rate_limited', race_id, f'HTTP {response.status_code}')
        return 'rate_limited'

    if response.status_code != 200:
        _record_failure('not_found', race_id, f'HTTP {response.status_code}')
        return 'not_found'

    response.encoding = 'EUC-JP'
    html_text = response.text

    if len(html_text) < 5000:
        _record_failure('not_found', race_id, f'size={len(html_text)}')
        return 'not_found'

    if 'データが存在しません' in html_text or 'お探しのページが見つかりませんでした' in html_text:
        _record_failure('not_found', race_id, 'page=no-data')
        return 'not_found'

    horse_link_count = html_text.count('/horse/')
    if horse_link_count < 3:
        _record_failure('not_found', race_id, f'horse_links={horse_link_count}')
        return 'not_found'

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html_text)

    return 'success'


print("✅ ダウンロード関数準備完了")


In [ ]:
# HTMLダウンロードの実行
print("📥 レースHTMLのダウンロードを開始します")

# ========================================================
# skip判定はディスク上のHTML有無のみで行う（download_race_html内で実施）
# 注: race_id は年・日付を表すIDではないため、Parquet の year 列での比較は
#     使用しない。Parquetのロードは「参考表示」用途のみ。
# ========================================================
from utils.data_loader import load_races
print("📥 既存Parquet（参考情報）を確認中...")
try:
    existing_races_df = load_races(from_github=True)
    print(f"📊 Parquet上のレース数（参考）: {len(existing_races_df):,}件")
except Exception as e:
    print(f"⚠️ Parquetの参照に失敗（無視して続行）: {e}")

print("⚠️ この処理には時間がかかる場合があります\n")

# 失敗理由カウンタをリセット
failure_reason_counts.clear()
failure_first_samples.clear()

# ダウンロード開始年の決定（当年のみ）
current_year = datetime.now().year
start_year = current_year
print(f"当年のみ更新: {start_year}年～現在\n")

generator = generate_race_ids(start_year=start_year)
stats = {'success': 0, 'exists': 0, 'not_found': 0, 'rate_limited': 0, 'error': 0}
last_success_race = None
total_processed = 0
first_failures_printed = 0
MAX_FIRST_FAILURES_TO_PRINT = 5

try:
    race_id = next(generator)

    while True:
        result = download_race_html(race_id)
        stats[result] = stats.get(result, 0) + 1
        total_processed += 1

        skip_signal = None

        if result == 'success':
            last_success_race = race_id
            skip_signal = 'race_found'
            print(f"✅ {race_id}: ダウンロード成功")
            time.sleep(0.5)
        elif result == 'exists':
            last_success_race = race_id
            skip_signal = 'race_found'
        elif result == 'not_found':
            # 最初の数件は失敗詳細を出力（原因切り分け用）
            if first_failures_printed < MAX_FIRST_FAILURES_TO_PRINT:
                last_reason = None
                # 直近の失敗詳細を取得
                for k, (rid, det) in failure_first_samples.items():
                    if rid == race_id:
                        last_reason = (k, det)
                        break
                if last_reason:
                    print(f"❌ {race_id}: not_found ({last_reason[1]})")
                else:
                    print(f"❌ {race_id}: not_found")
                first_failures_printed += 1
            race_num = int(race_id[-2:])
            if race_num == 1:
                day = int(race_id[-4:-2])
                if day == 1:
                    skip_signal = 'skip_kaisai'
                else:
                    skip_signal = 'skip_day'
        elif result == 'rate_limited':
            print(f"⚠️ {race_id}: rate_limited (429/503) → 5秒待機")
            time.sleep(5.0)
        elif result == 'error':
            if first_failures_printed < MAX_FIRST_FAILURES_TO_PRINT:
                for k, (rid, det) in failure_first_samples.items():
                    if rid == race_id and k == 'error':
                        print(f"❌ {race_id}: error ({det})")
                        break
                first_failures_printed += 1

        if total_processed % 100 == 0:
            print(f"\n📊 進捗: {total_processed}件処理")
            print(f"   成功: {stats['success']}, 既存: {stats['exists']}")
            print(f"   未発見: {stats['not_found']}, rate制限: {stats['rate_limited']}, エラー: {stats['error']}\n")

        race_id = generator.send(skip_signal)

except StopIteration:
    print("\n✅ ダウンロード完了（全race_id処理済み）")

print(f"\n📊 最終統計:")
print(f"   処理件数: {total_processed}件")
print(f"   新規ダウンロード: {stats['success']}件")
print(f"   既存スキップ(ディスクに存在): {stats['exists']}件")
print(f"   未発見: {stats['not_found']}件")
print(f"   rate制限(429/503): {stats['rate_limited']}件")
print(f"   エラー: {stats['error']}件")
if last_success_race:
    print(f"   最新レース: {last_success_race}")

# 失敗理由の内訳
if failure_reason_counts:
    print(f"\n🔍 失敗理由の内訳（詳細）:")
    detail_counts = {}
    for k, (rid, det) in failure_first_samples.items():
        detail_counts.setdefault(k, []).append((rid, det))
    # サンプルを表示
    print(f"   各カテゴリの最初のサンプル:")
    for k, samples in detail_counts.items():
        for rid, det in samples[:1]:
            print(f"      [{k}] race_id={rid} 詳細={det}")

# 期待される動作との乖離をチェック
if stats['success'] == 0 and stats['not_found'] > 50:
    print(f"\n⚠️ 新規ダウンロードが0件で、未発見が{stats['not_found']}件以上あります。")
    print(f"   netkeiba が Colab IP からのアクセスをレート制限/ブロックしている可能性があります。")
    print(f"   上記「失敗理由の内訳」を確認してください。")
